# 参考
- [NLP关键词提取必备：从TFIDF到KeyBert范式原理优缺点与开源实现](https://mp.weixin.qq.com/s/H2aRyF3tNZCCOxa4Oy2JMw)
- [KeyBert、TextRank等九种本文关键词提取算法（KPE）原理及代码实现](https://blog.csdn.net/weixin_43734080/article/details/124656853)
- [TF-IDF 原理与实现](https://zhuanlan.zhihu.com/p/97273457)

# 基于词袋加权的TFIDF算法

In [3]:
import jieba.analyse
text = "美国总统拜登当地时间26日又语出惊人。据路透社报道，拜登当天在波兰的演 讲中，称俄罗斯总统普京“不能继续掌权了”。\
    白宫官员随后出来解释，克里姆林宫同日 也作出回应，称“这不是由拜登决定的”。报道称，在华沙皇家城堡发表的讲话中，拜登在\
    谴责普京后表示“看在上帝的份上，这个人不能继续掌权了”。对此，路透社表示，该言论引发了华盛顿方面对局势升级的担忧，美国\
    一直避免直接对乌克兰进行军事干预，并明确表示不支持政权更迭。随后，一名白宫官员对拜登这番话进行解释，称拜登的言论并不\
    代表华盛顿政策的转变，其目的是让世界为乌克兰问题上的长期冲突做好准备。该官员称，拜登意为“不能 允许普京对其邻国或该地区\
    行使权力”，而不是在讨论普京在俄罗斯的权力或俄 罗斯的政权 更迭问题。同日，路透社称，被问及拜登这一言论时，俄罗斯总统新闻\
    秘书佩斯科夫作出回应，称“这不是由拜登决定的。俄罗斯总统是由俄罗斯人选举产生的”。此前 ，俄罗斯安全会议副主席梅德韦杰夫接受\
    俄媒采访时曾表示，俄特别军事行动既定目标包括实现乌克兰去军事化、去纳粹化、成为中立国家、不奉行反俄政策。俄开展特别军事行动\
    首先是因为其设定的目标未能通过外交方式实现。他表示，社会调查显示，四分之三的俄罗斯民众支持在乌开展特别军事行动。西方国家企\
    图通过制裁影响俄民众，煽动他们反对政府，结果适得其反。"
jieba.analyse.extract_tags(text, topK=20, withWeight=True, allowPOS=('ns', 'n', 'vn', 'v','nr', 'nt'))

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\Lenovo\AppData\Local\Temp\jieba.cache
Loading model cost 0.563 seconds.
Prefix dict has been built successfully.


[('俄罗斯', 0.3118400196635),
 ('普京', 0.2536227751751428),
 ('路透社', 0.19182680917714287),
 ('军事行动', 0.186833445036),
 ('乌克兰', 0.17846836328142857),
 ('总统', 0.17652694020771428),
 ('言论', 0.16598184767057142),
 ('表示', 0.1397577380592857),
 ('掌权', 0.13398412964185713),
 ('白宫', 0.12738672900385714),
 ('官员', 0.12492367635407142),
 ('华盛顿', 0.11253185798585714),
 ('回应', 0.10643153185471428),
 ('民众', 0.10102807971899999),
 ('中立国家', 0.09433950336714286),
 ('政权', 0.09042455812585715),
 ('斯科夫', 0.08649227273357143),
 ('开展', 0.08633691124385715),
 ('不能', 0.08550353216785714),
 ('份上', 0.08539119644928571)]

In [1]:
# 输入语料库
corpus = ['this is the first document',
        'this is the second second document',
        'and the third one',
        'is this the first document']

words_list = list()
for i in range(len(corpus)):
    words_list.append(corpus[i].split(' '))
### 第一种 gensim
from gensim import corpora
# 赋给语料库中每个词(不重复的词)一个整数id
dic = corpora.Dictionary(words_list)
# print(dic.token2id)
# 元组中第一个元素是词语在词典中对应的id，第二个元素是词语在文档中出现的次数
new_corpus = [dic.doc2bow(words) for words in words_list]

# 训练模型并保存
from gensim import models
tfidf = models.TfidfModel(new_corpus)
tfidf_vec = []
for i in range(len(corpus)):
    string = corpus[i]
    string_bow = dic.doc2bow(string.lower().split())
    string_tfidf = tfidf[string_bow]
    tfidf_vec.append(string_tfidf)
# 输出 词语id与词语tfidf值
print(tfidf_vec)

# ### 第二种 sklearn
# from sklearn.feature_extraction.text import TfidfVectorizer
# tfidf_vec = TfidfVectorizer()
# tfidf_matrix = tfidf_vec.fit_transform(corpus)
# # 得到语料库所有不重复的词
# print(tfidf_vec.get_feature_names())
# # 得到每个单词对应的id值
# print(tfidf_vec.vocabulary_)
# # 得到每个句子所对应的向量，向量里数字的顺序是按照词语的id顺序来的
# print(tfidf_matrix.toarray())

{'document': 0, 'first': 1, 'is': 2, 'the': 3, 'this': 4, 'second': 5, 'and': 6, 'one': 7, 'third': 8}
[[(0, 0.33699829595119235), (1, 0.8119707171924228), (2, 0.33699829595119235), (4, 0.33699829595119235)], [(0, 0.10212329019650272), (2, 0.10212329019650272), (4, 0.10212329019650272), (5, 0.9842319344536239)], [(6, 0.5773502691896258), (7, 0.5773502691896258), (8, 0.5773502691896258)], [(0, 0.33699829595119235), (1, 0.8119707171924228), (2, 0.33699829595119235), (4, 0.33699829595119235)]]


# 考虑词关联网络的TextRank算法

In [5]:
import jieba.analyse
text = "美国总统拜登当地时间26日又语出惊人。据路透社报道，拜登当天在波兰的演 讲中，称俄罗斯总统普京“不能继续掌权了”。\
    白宫官员随后出来解释，克里姆林宫同日 也作出回应，称“这不是由拜登决定的”。报道称，在华沙皇家城堡发表的讲话中，拜登在\
    谴责普京后表示“看在上帝的份上，这个人不能继续掌权了”。对此，路透社表示，该言论引发了华盛顿方面对局势升级的担忧，美国\
    一直避免直接对乌克兰进行军事干预，并明确表示不支持政权更迭。随后，一名白宫官员对拜登这番话进行解释，称拜登的言论并不\
    代表华盛顿政策的转变，其目的是让世界为乌克兰问题上的长期冲突做好准备。该官员称，拜登意为“不能 允许普京对其邻国或该地区\
    行使权力”，而不是在讨论普京在俄罗斯的权力或俄 罗斯的政权 更迭问题。同日，路透社称，被问及拜登这一言论时，俄罗斯总统新闻\
    秘书佩斯科夫作出回应，称“这不是由拜登决定的。俄罗斯总统是由俄罗斯人选举产生的”。此前 ，俄罗斯安全会议副主席梅德韦杰夫接受\
    俄媒采访时曾表示，俄特别军事行动既定目标包括实现乌克兰去军事化、去纳粹化、成为中立国家、不奉行反俄政策。俄开展特别军事行动\
    首先是因为其设定的目标未能通过外交方式实现。他表示，社会调查显示，四分之三的俄罗斯民众支持在乌开展特别军事行动。西方国家企\
    图通过制裁影响俄民众，煽动他们反对政府，结果适得其反。"
jieba.analyse.textrank(text, topK=20, withWeight=True, allowPOS=('ns', 'n', 'vn', 'v','nr', 'nt'))

[('俄罗斯', 1.0),
 ('表示', 0.814487158448448),
 ('普京', 0.6186980825111877),
 ('总统', 0.5566981083583048),
 ('乌克兰', 0.5444185252285827),
 ('不能', 0.4551057696773949),
 ('实现', 0.43825213813627867),
 ('官员', 0.41551314966922487),
 ('华盛顿', 0.4099423976893444),
 ('军事行动', 0.4084047245015465),
 ('言论', 0.3931025309127227),
 ('民众', 0.367154284143075),
 ('进行', 0.3599413337375733),
 ('政策', 0.2874849474541063),
 ('问题', 0.28241812756236145),
 ('未能', 0.27536496876800404),
 ('路透社', 0.27518649210713825),
 ('回应', 0.2750299079863733),
 ('作出', 0.27353643806493844),
 ('继续', 0.27137798625510273)]

# LDA